## CS431/631 Data Intensive Distributed Computing
### Winter 2025 - Assignment 4
---

**Please edit this (text) cell to provide your name and UW student ID number!**
* **Name:** Ajun Jo
* **ID:** 20960638

Spark is not installed in Colab so we have to install it ourself. This will take a minute to finish. If you're using this on your own machine the following might not work, and you will have to install Spark yourself.

In [1]:
!apt-get update -qq > /dev/null
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!curl -Os https://student.cs.uwaterloo.ca/~cs451/spark/spark-3.4.3-bin-hadoop3.tgz
!tar xzf spark-3.4.3-bin-hadoop3.tgz
!pip install -q findspark

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


Now that you installed Spark and Java in Colab, it is time to set the environment path which enables you to run Pyspark in your Colab environment. On your own system you won't need to run the next box, but on Colab you must.

In [2]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.4.3-bin-hadoop3"

Now you will be able to create the SparkContext object needed to run Spark code:

In [3]:
import findspark
findspark.init()

from pyspark import SparkContext
sc = SparkContext(appName="YourTest", master="local[*]")

If you are running on colab, you can run the next code block to create a clickable link to open the SparkUI. (If running on your own machine, then the previous cell should have given you a link that will work on your local machine, but if not, try localhost:4040)

In [4]:
from google.colab import output
output.serve_kernel_port_as_window(4040, path='/jobs/index.html')
# This will create a link below. You must click the link, do not copy & paste the URL as that's the "local" URL and won't work on your machine

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

---
#### Overview
For this assignment, you will be using Python and Spark to perform spam detection.   You will need to perform two tasks.   The first is to build spam prediction models, using training data sets and stochastic gradient descent (SGD).   The second is to use these models to predict whether the documents in a test data set are spam.
The stochastic gradient descent technique that you will be using is based on [a paper](http://arxiv.org/abs/1004.5168) by Cormack, Smucker and Clarke.

#### Training a Spam Classification Models
To build a spam classification model, you will start with a training data set.   Each instance in the training set represents a single document, and is labeled to indicate whether that document should be considered to be spam or ham.
An instance looks like this:
```
clueweb09-en0094-20-13546 spam 387908 697162 426572 161118 688171 ...
```
The first field, `clueweb09-en0094-20-13546`, is the (unique) document name.   The second field is the label, indicating whether the document should be considered spam (as in this example) or ham.   The remaining fields are integers representing *features* present in the document.   In this case, the features are hashed byte 4-grams, represented as integers.   Each training data set is stored as a text file, with one instance per line.   The training files  are:
* `spam.train.group_x.txt`   (25 MB)
* `spam.train.group_y.txt`   (20 MB)
* `spam.train.britney.txt`   (766 MB)

Now let's download the spamminess module and the training traces we will use in this assignment. This will take a few minutes. The ls command at the end shows the files we have in this directory. Make sure all files are here now.

In [5]:
!wget -q https://student.cs.uwaterloo.ca/~cs451/W20/content/cs431/spamminess.py
!wget -q https://www.student.cs.uwaterloo.ca/~cs451/spam/spam.train.group_x.txt.bz2
!wget -q https://www.student.cs.uwaterloo.ca/~cs451/spam/spam.train.group_y.txt.bz2
!wget -q https://www.student.cs.uwaterloo.ca/~cs451/spam/spam.train.britney.txt.bz2

!bunzip2 spam.train.group_x.txt.bz2
!bunzip2 spam.train.group_y.txt.bz2
!bunzip2 spam.train.britney.txt.bz2
!ls

sample_data		spam.train.group_x.txt	 spark-3.4.3-bin-hadoop3.tgz
spamminess.py		spam.train.group_y.txt
spam.train.britney.txt	spark-3.4.3-bin-hadoop3



---
### Important

The questions that follow ask you to implement functions whose prototypes are given to you. Do _**not**_ change the prototypes of the functions. Do _**not**_ write code outside of the functions.

You may use specific cells, identified by `# Your tests here`, for test purposes. Code in these cells will *not* be executed when marking your assignment.

---

#### Question 1 ( 5/20 marks)

Your first task is to write a sequential SGD model trainer in Python (no Spark).   For our purposes, a model associates a *weight* with each feature.   The model trainer decides what these weights should be, based on the training instances.  Since you are going to be writing a model trainer based on SGD, the trainer should behave like this:
```
for each training instance T
   predict whether T is spam or ham using the weights of the current model
   update the model weights by comparing T's predicted label with its actual label
```
Of course, the important part is how to update the model.

In [the paper](http://arxiv.org/abs/1004.5168), the model is used to assign a "spamminess" score to a document.   Documents with positive spamminess are predicted to be spam.   Those with negative spamminess are predicted to be ham.  The spamminess of a document $D$ is simply the sum of the weights (from the model) of each of the document's features:
\begin{equation}
spamminess(D) = \sum_{f \in D}{w(f)}
\end{equation}
where $w(f)$ is the weight assocated with feature $f$.

The Python module `spamminess.py` defines a function `spamminess(F,W)` which computes this quantity.   This function takes two arguments, `F` and `W`.  `F` is a list of features (integers) associated to the document whose spamminess you want to compute, and `W` is a dictionary representing the current model.  `W` maps features ($f$) to their weights ($w(f)$) under the model.

In the cell below, you will find partial pseudo-code that shows how to implement the SGD model trainer defined by Cormack, Smucker, and Clarke.   It reads the training instances one at a time from one of the training files, and uses them to adjust the model weights.   Your job is to turn this pseudo-code into actual runnable Python code that can
be used to learn a model from any one of the training files. Implement the function `sequential_SGD()` that takes as input a model (`w`), the training dataset and a value for the update parameter `delta`, and returns the trained model.

In [20]:
# A4Q1 (fixed)
from spamminess import spamminess
from math import exp

def sequential_SGD(model, training_dataset='spam.train.group_x.txt', delta=0.002):
    with open(training_dataset, encoding="utf-8") as f:
        for raw in f:
            s = raw.strip()
            if not s or s.startswith("#"):
                continue

            toks = s.split()
            # toks[0] = docid, toks[1] = label, toks[2:] = features
            t = 1.0 if toks[1].lower().startswith("spam") else 0.0
            F = [int(u) for u in toks[2:] if u.lstrip("-").isdigit()]

            score = spamminess(F, model)
            prob  = 1.0 / (1.0 + exp(-score))
            step  = (1.0 - prob) * delta if t == 1.0 else -prob * delta

            for ftr in F:
                model[ftr] = model.get(ftr, 0.0) + step

    return model


In [22]:
# ---------------- Q1 tests ----------------
import os
from math import exp

def _write(path, text):
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)

def _approx(a, b, tol=1e-12):
    return abs(a - b) <= tol

# Test 1: exact updates on a 2-line toy set (docid label feat1 feat2 ...)
δ = 0.002
toy1 = "docA spam 1 2\n" \
       "docB ham  2 3\n"
_write("toy_q1_1.txt", toy1)

m = {}
ret = sequential_SGD(m, training_dataset="toy_q1_1.txt", delta=δ)

# expected values
step1 = (1.0 - 0.5) * δ                  # 0.001
p2    = 1.0 / (1.0 + exp(-step1))
step2 = -p2 * δ

assert ret is m, "Function should return the same dict object it mutates"
assert set(m.keys()) == {1, 2, 3}, "Expected features {1,2,3}"
assert _approx(m[1], step1), "w[1] after line 1"
assert _approx(m[2], step1 + step2), "w[2] after line 2"
assert _approx(m[3], step2), "w[3] after line 2"

# Test 2: parsing robustness — comments, blanks, and non-numeric tokens ignored
toy2 = """
# header line
docX spam 10 foo 20

docY ham  10  bar
""".lstrip()
_write("toy_q1_2.txt", toy2)

m2 = {}
sequential_SGD(m2, training_dataset="toy_q1_2.txt", delta=δ)


p = 1.0 / (1.0 + exp(-0.001))
step = -p * δ
assert set(m2.keys()) == {10, 20}, "Only integer features should be learned"
assert _approx(m2[20], 0.001), "w[20] should only get spam step"
assert _approx(m2[10], 0.001 + step), "w[10] should combine spam and ham steps"

# Test 3: idempotence on empty file — no changes
_write("toy_q1_empty.txt", "")
before = dict(m2)
sequential_SGD(m2, training_dataset="toy_q1_empty.txt", delta=δ)
assert m2 == before, "Empty input must not change the model"

print("All Q1 tests passed.")


All Q1 tests passed.


#### Question 2 (5/20 marks)

Next, you should try implementing a Spark version of the SGD model trainer.   Your Spark implementation should read a training file, train the model, and then output the model to the `models` folder.  The model output file that you generate should list the weight associated with each feature, with one feature per line, like this:
```
(802123, 0.0009858585991850937)
(438450, 4.267897922108138e-05)
(271525, 0.0013133437007968654)
(92853, 0.0004300009932503611)
```

Use Spark's `saveAsTextFile` action to output your model.   For example, if you are training a model for the group_x training set, use `saveAsTextFile("models/group_x_model")`.   This will actually cause Spark to create a folder called `group_x_model`.   In the folder, there will be files with names like `part-00000` that contain the actual output data.  When you use `saveAsTextFile`, Spark will generate one `part-xxxxx` file for each partition of the RDD that you are writing out.   In this case, you should have only a single partition (for the reason described below), so there should be only one `part-xxxxx` file.

Training the SGD model is an inherently sequential task, since the training instances update the model one at a time, and each instance's spamminess is computed using the model produced by that instance's predecessors.   This means that the only part of the training that you can parallelize using Spark is the parsing of the input file.   Once the input is parsed, your Spark implementation will have to force all of the instances into a single partition, and then apply the training function to the entire partition.   To see whether you are getting sensible results, you can compare the model you learn with Spark to the one that you learned with your sequential Python program from Question 1.

Remember that training should occur entirely in Spark.  The training instances should never come into your driver program.

Implement the function `spark_SGD()` below that takes as input the path to the training dataset, an output path `output_model` and a value for the update parameter `delta`, and writes the trained model to `output_model` using Spark's `saveAsTextFile`. You can use it to generate models from all three of the training files, leaving the results in your models folder. For this assignment, you will be using Spark's original RDD interface, rather than the DataFrame interface.

Hint: You need to move all of the data into one partition and then use mapPartition to train the model.


In [23]:
# A4Q2
from spamminess import spamminess
from math import exp
import os, shutil

def _train_partition(lines_iter, delta):
    w = {}
    for line in lines_iter:
        s = line.strip()
        if not s or s.startswith("#"):
            continue

        toks = s.split()
        if len(toks) < 2:
            continue  # malformed line; skip

        # toks[0] = docid, toks[1] = label, toks[2:] = features
        t = 1.0 if toks[1].lower().startswith("spam") else 0.0
        feats = [int(u) for u in toks[2:] if u.lstrip("-").isdigit()]

        score = spamminess(feats, w)
        p = 1.0 / (1.0 + exp(-score))
        step = (1.0 - p) * delta if t == 1.0 else -p * delta
        for ftr in feats:
            w[ftr] = w.get(ftr, 0.0) + step

    return iter(w.items())  # (feature, weight)

def spark_SGD(training_dataset='spam.train.group_x.txt',
              output_model='models/group_x_model',
              delta=0.002):
    # clean previous output
    if os.path.isdir(output_model):
        shutil.rmtree(output_model)

    # single partition so the per-partition loop is sequential and produces one part-* file
    rdd = sc.textFile(training_dataset).coalesce(1)
    model_pairs = rdd.mapPartitions(lambda it: _train_partition(it, delta))

    # save strings, not tuples
    (model_pairs
        .map(lambda kv: f"({int(kv[0])}, {float(kv[1])})")
        .coalesce(1)
        .saveAsTextFile(output_model))


#### Question 3 (5/20 marks)

When you train a model using SGD, the model you get depends on the order in which you handle the training instances.  To see this in action, try using the Spark SGD trainer you implemented for Question 2 to train a model from the group_x training set, but with the instances processed in a different order.  

To do this, re-implement your trainer from Question 2 so that it will randomly reorder the training instances before using them to update the model. One way to shuffle the training instances is to assign a random sort key to each training instance as you read it from the input file, and then sort the instances using the random sort key.

Be sure that Spark is doing the work of shuffling the training instances.   Do not load the training instances into your driver program and sort them there.

Implement the function `spark_shuffled_SGD` below that takes as input the path to the training dataset, an output path `output_model` and a value for the update parameter `delta`, shuffles the training instances using the method described above and writes the trained model to `output_model` using Spark's `saveAsTextFile`.

Once you have implemented the shuffled trainer, train a model using shuffled group_x training instances, and compare the resulting model with group_x model you learned without shuffling.  It is up to you how to do this comparision.  At a minimum, compare features with the highest weights in each model to see if they are similar. You can also use the classifier in next question to classify documents using the two models, and compare results.


In [25]:
# A4Q3
# A4Q3 (fixed)
from spamminess import spamminess
from math import exp
import os, shutil

def _train_partition(lines_iter, delta):
    w = {}
    for line in lines_iter:
        s = line.strip()
        if not s or s.startswith("#"):
            continue
        toks = s.split()
        if len(toks) < 2:
            continue  # malformed

        # toks[0] = docid, toks[1] = label, toks[2:] = features
        t = 1.0 if toks[1].lower().startswith("spam") else 0.0
        feats = [int(u) for u in toks[2:] if u.lstrip("-").isdigit()]
        if not feats:
            continue

        score = spamminess(feats, w)
        p = 1.0 / (1.0 + exp(-score))
        step = (1.0 - p) * delta if t == 1.0 else -p * delta
        for ftr in feats:
            w[ftr] = w.get(ftr, 0.0) + step
    return iter(w.items())  # (feature, weight)

def spark_shuffled_SGD(training_dataset='spam.train.group_x.txt',
                       output_model='models/group_x_model_shuffled',
                       delta=0.002, seed=42):
    if os.path.isdir(output_model):
        shutil.rmtree(output_model)

    rdd = sc.textFile(training_dataset)

    # Deterministic shuffle by index -> one partition for sequential training
    def _rand_from_index(i, seed=seed):
        a, c, m = 1103515245, 12345, 2**31
        return ((a * (i + seed) + c) % m) / float(m)

    shuffled = (rdd
                .zipWithIndex()
                .map(lambda li: (_rand_from_index(li[1]), li[0]))
                .sortByKey(numPartitions=1)   # shuffle in Spark; produce 1 partition
                .values())

    model_pairs = shuffled.mapPartitions(lambda it: _train_partition(it, delta))

    (model_pairs
        .map(lambda kv: f"({int(kv[0])}, {float(kv[1])})")  # save strings, not tuples
        .coalesce(1)
        .saveAsTextFile(output_model))


In [26]:
# Your tests here
spark_shuffled_SGD(output_model='models/group_x_model_shuffled')


AssertionError: Different seeds should produce different weights

#### Question 4 (5/20  marks)

Last but not least, you should write a Spark program that can be used to classify documents as spam or ham, using the classification models you produced.

The test data, i.e., the document instances that you should classifiy, are located in `spam.test.qrels.txt`. Run the following block to download this trace. This will take a few minutes.

In [14]:
!wget -q https://www.student.cs.uwaterloo.ca/~cs451/spam/spam.test.qrels.txt.bz2
!bunzip2 spam.test.qrels.txt.bz2
!ls

empty.txt    spamminess.py	     spam.train.group_y.txt	  toy2.txt
models	     spam.test.qrels.txt     spark-3.4.3-bin-hadoop3	  toy_spark.txt
__pycache__  spam.train.britney.txt  spark-3.4.3-bin-hadoop3.tgz
sample_data  spam.train.group_x.txt  toy1.txt



Each line in this file represents a document that needs to be classified as spam or ham.  The format of this file is identical to the format of the files that hold the training instances.

Implement the function `spark_classify` below that will load a model (from a specified folder under `models`), classify all of the instances in a given test data file (`spam.test.qrels.txt` by default) using that model, and then output the results in the folder `results_path` using Spark's `saveAsTextFile`.   The contents of the output file should look like this:
```
(clueweb09-en0000-00-00142,spam,2.601624279252943,spam)
(clueweb09-en0000-00-01005,ham,2.5654162439491004,spam)
(clueweb09-en0000-00-01382,ham,2.5893946346394188,spam)
```
Each line of the output represents one test instance.   The first two fields are the document ID and the test label.  These are just copied from the test data.   The third field is the spamminess score of the document, produced by the spamminess function using the model you are classifying with.   The fourth field is the spam/ham prediction made by the model.

Of course, your spam/ham classifier must **not** use the test label from the input when making its prediction.  The test labels are the "ground truth" against which your predictions are being compared.   Using them to make predictions would defeat the whole purpose of model-based classification.

Make sure that classification of the test instances is done by Spark, not by your driver program.  Do ***not*** load the test instances or classification results into your driver program. You are however allowed to load the model weights into your driver program to distribute them as side data.
Unlike model training, classification is easily parallelizable, since each document is classified independently.

In [27]:
# A4Q4
from spamminess import spamminess
import os, shutil

def _parse_model_line(s):
    # "(123, 0.45)" -> (123, 0.45)
    s = s.strip()
    if not s:
        return None
    if s[0] == "(" and s[-1] == ")":
        s = s[1:-1]
    k, v = s.split(",", 1)
    return (int(k.strip()), float(v.strip()))

def spark_classify(input_model='models/group_x_model',
                   test_dataset='spam.test.qrels.txt',
                   results_path='results/test_qrels'):
    # Remove previous results so saveAsTextFile succeeds
    if os.path.isdir(results_path):
        shutil.rmtree(results_path)

    # Load learned weights on the driver (allowed) and broadcast as side data
    model_lines = sc.textFile(os.path.join(input_model, "part-*"))
    weights = dict(
        model_lines.map(_parse_model_line)
                   .filter(lambda x: x is not None)
                   .collect()
    )
    bw = sc.broadcast(weights)

    # Classify entirely in Spark; do not bring test data/results to the driver
    def _classify(line: str):
        s = line.strip()
        if not s or s.startswith("#"):
            return None
        toks = s.split()
        # toks[0]=docid, toks[1]=true label, toks[2:]=features
        docid, true_lbl = toks[0], toks[1]
        feats = [int(u) for u in toks[2:] if u.lstrip("-").isdigit()]
        score = spamminess(feats, bw.value)
        pred  = "spam" if score > 0.0 else "ham"
        return f"({docid},{true_lbl},{score},{pred})"

    (sc.textFile(test_dataset)
       .map(_classify)
       .filter(lambda x: x is not None)
       .coalesce(1)                      # single output part-xxxxx
       .saveAsTextFile(results_path))


We have developed a program that can be used to evaluate your classification results.  Run the next block to download this program.

In [15]:
!wget -q https://student.cs.uwaterloo.ca/~cs451/content/cs431/compute_spam_metrics.c
!wget -q https://student.cs.uwaterloo.ca/~cs451/content/cs431/spam_eval.sh

Now compile this program.

In [16]:
!gcc -w -O2 -o compute_spam_metrics compute_spam_metrics.c -lm

 Given your ouput file, in the proper format, it will compute the area under the receiver operating curve (ROC).   This is a common way to characterize classifier error.    The lower this score, the better.   The evaluation program should produce one line of output, like this
```
1-ROCA%: 17.25
```

Use your classifier to classify the test instances using each of the three classification models that you produced, which should result in three different output files.   Then, in the cell below,
use the evaluation program to evaluate your results.


In [36]:
# Your tests here
#  Run the evaluation program like this, after first replacing "output-file"
#  with the name of the folder that holds your classifier's output
!bash spam_eval.sh results/group_x_results
!bash spam_eval.sh results/group_y_results
!bash spam_eval.sh results/britney_results

1-ROCA%: 17.25
1-ROCA%: 12.82
1-ROCA%: 15.96


---
Don't forget to save your workbook!   When you are finished and you are ready to submit your assignment, download your notebook file (.ipynb) from the hub to your machine, and then follow the submission instructions in the assignment.